# 03 · Backtest Review

This notebook demonstrates how to:
1. Load signals and run the TradeSimulator across all symbols
2. Compute PerformanceMetrics (win rate, avg R, profit factor, Sharpe)
3. Visualise equity curve, R distribution, exit reasons
4. Break down performance by setup type and trigger type

In [ ]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from martin_quant.backtest.trade_simulator import TradeSimulator, TradeSimulatorConfig
from martin_quant.backtest.performance import compute_performance

SIGNALS_PATH = Path('../data/outputs/signals.parquet')
OHLCV_DIR    = Path('../data/outputs/daily/')
print('Paths ready')

## Load Signals & OHLCV

In [ ]:
# Signals are TriggerSignal objects saved as parquet by daily_scan.py
# For notebook testing, we build dummy signals from the CSV if parquet missing
if SIGNALS_PATH.exists():
    signals_df = pd.read_parquet(SIGNALS_PATH)
    print(f'Loaded {len(signals_df)} signals')
else:
    print('No signals parquet found. Run daily_scan.py first.')
    signals_df = pd.DataFrame()

ohlcv_map = {}
for path in OHLCV_DIR.glob('*_daily.parquet'):
    sym = path.stem.replace('_daily', '').upper()
    df  = pd.read_parquet(path)
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    ohlcv_map[sym] = df
print(f'Loaded OHLCV for {len(ohlcv_map)} symbols')

## Run TradeSimulator

In [ ]:
# Rebuild TriggerSignal objects from the signals DataFrame
from martin_quant.core.datatypes import TriggerSignal
from martin_quant.core.enums import TriggerType, SetupType

def row_to_trigger(row: pd.Series):
    return TriggerSignal(
        symbol=row['symbol'],
        timestamp=row['timestamp'],
        trigger_type=TriggerType(row['trigger_type']),
        timeframe=row.get('timeframe', '15m'),
        direction=row.get('direction', 'long'),
        entry_price=row.get('entry_price'),
        stop_price=row.get('stop_price'),
        target_price=row.get('target_price'),
        linked_setup_type=SetupType(row['linked_setup_type']) if pd.notna(row.get('linked_setup_type')) else None,
    )

signals = [row_to_trigger(r) for _, r in signals_df.iterrows()] if not signals_df.empty else []
print(f'Reconstructed {len(signals)} TriggerSignal objects')

sim_cfg  = TradeSimulatorConfig(max_holding_days=20, trail_mode='ema9', slippage_pct=0.05)
simulator = TradeSimulator(config=sim_cfg)
trades    = simulator.run_backtest(signals=signals, ohlcv_daily_map=ohlcv_map, shares_per_trade=100)
print(f'Simulated trades: {len(trades)}')

## Performance Summary

In [ ]:
metrics = compute_performance(trades)
perf_df = pd.DataFrame([metrics.to_dict()])

print('===== PERFORMANCE SUMMARY =====')
print(f"Total trades  : {metrics.total_trades}")
print(f"Win rate      : {metrics.win_rate:.1%}")
print(f"Avg R         : {metrics.avg_r:.2f}")
print(f"Avg Win R     : {metrics.avg_win_r:.2f}")
print(f"Avg Loss R    : {metrics.avg_loss_r:.2f}")
print(f"Profit Factor : {metrics.profit_factor:.2f}")
print(f"Expectancy R  : {metrics.expectancy_r:.3f}")
print(f"Sharpe (R)    : {metrics.sharpe_ratio:.3f}")
print(f"Max Drawdown  : {metrics.max_drawdown_pct:.1f}%")
print(f"Total PnL     : ${metrics.total_pnl:,.0f}")

## Equity Curve

In [ ]:
if trades:
    trades_df   = pd.DataFrame([t.to_dict() for t in trades])
    trades_df['exit_date'] = pd.to_datetime(trades_df['exit_date'])
    trades_df   = trades_df.sort_values('exit_date')
    equity_curve = trades_df['pnl'].cumsum()

    fig, axes = plt.subplots(1, 2, figsize=(16, 4))

    # Equity curve
    axes[0].plot(trades_df['exit_date'], equity_curve, lw=1.5, color='steelblue')
    axes[0].axhline(0, color='black', lw=0.8, ls='--')
    axes[0].set_title('Cumulative PnL ($)')
    axes[0].set_xlabel('Date')
    axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

    # R distribution
    axes[1].hist(trades_df['r_multiple'], bins=20, edgecolor='black',
                 color=['tomato' if r < 0 else 'mediumseagreen' for r in trades_df['r_multiple']])
    axes[1].axvline(0, color='black', lw=0.8, ls='--')
    axes[1].set_title('R-Multiple Distribution')
    axes[1].set_xlabel('R Multiple')
    axes[1].set_ylabel('Count')

    plt.tight_layout()
    plt.show()

## Exit Reason Breakdown

In [ ]:
if trades:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # By exit reason
    exit_counts = trades_df['exit_reason'].value_counts()
    axes[0].bar(exit_counts.index, exit_counts.values, color='steelblue', edgecolor='black')
    axes[0].set_title('Trades by Exit Reason')
    axes[0].set_ylabel('Count')

    # By setup type
    if 'setup_type' in trades_df.columns and trades_df['setup_type'].nunique() > 1:
        setup_pnl = trades_df.groupby('setup_type')['pnl'].sum()
        colors = ['mediumseagreen' if v >= 0 else 'tomato' for v in setup_pnl.values]
        axes[1].bar(setup_pnl.index, setup_pnl.values, color=colors, edgecolor='black')
        axes[1].set_title('PnL by Setup Type')
        axes[1].set_ylabel('Total PnL ($)')
    else:
        axes[1].set_visible(False)

    plt.tight_layout()
    plt.show()

## Trade Log

In [ ]:
if trades:
    cols = ['symbol','entry_date','exit_date','entry_price','exit_price',
            'shares','pnl','r_multiple','exit_reason','holding_days','setup_type']
    cols = [c for c in cols if c in trades_df.columns]
    trades_df.sort_values('r_multiple', ascending=False)[cols].head(20)